# V7 — MSAPSO con parada anticipada (demo en Colab)
Este notebook corre el algoritmo de la V7 sobre los 4 casos reales de sombreado parcial y lo compara con el MSAPSO completo del paper (25 pasadas, 175 mediciones).
**Idea:** igual que el MSAPSO, pero se detiene cuando las 7 partículas ya convergieron (dispersión std(x)/Isc < 0.02). Sin entrenamiento ni modelos.

**Antes de empezar:** comprime la carpeta `Proyecto Integrador Paneles_V7` en un `.zip` (o súbela a tu Drive) y ejecuta las celdas en orden.

## 1. Cargar el proyecto

In [ ]:
# Opción A: subir el .zip desde tu computadora
from google.colab import files
subido = files.upload()                      # elige el zip de la carpeta V7
import zipfile, os
nombre = next(iter(subido))
zipfile.ZipFile(nombre).extractall("v7")
RAIZ = next(r for r, d, f in os.walk("v7") if "msapso_early_V7.py" in f)
%cd {RAIZ}
# Opción B (Drive): from google.colab import drive; drive.mount('/content/drive'); %cd '/content/drive/MyDrive/Proyecto Integrador Paneles_V7'

In [ ]:
!pip -q install numpy scipy matplotlib

## 2. Una corrida: V7 vs MSAPSO completo en un caso real

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from msapso_early_V7 import *

caso = 2                                     # 0..3 (casos reales 1..4)
curva = scenarios()[0][caso]

def correr(regla, **kw):
    pl = StaticPlant(curva)
    make_msapso_early(regla, **kw)(pl, curva, np.random.default_rng(0))
    return pl

v7   = correr(Rule("disp", delta=0.02, t_min=3))   # parada anticipada
full = correr(Rule("none"))                         # MSAPSO completo (25 pasadas)
pmax = curva.P.max()
for nombre, pl in (("V7", v7), ("MSAPSO completo", full)):
    print(f"{nombre:16s} {len(pl.log_I):3d} mediciones | mejor P = {max(pl.log_Ptrue):.2f} W ({100*max(pl.log_Ptrue)/pmax:.2f} % del máximo)")

plt.figure(figsize=(8, 4)); plt.plot(curva.I, curva.P, lw=2, label="curva P-I")
plt.scatter(full.log_I, full.log_Ptrue, s=14, c="lightgray", label="MSAPSO completo (175)")
plt.scatter(v7.log_I, v7.log_Ptrue, s=14, c="crimson", label="V7 (70)")
plt.xlabel("Corriente (A)"); plt.ylabel("Potencia (W)"); plt.legend(); plt.show()

## 3. Los 4 casos reales, con ruido de medición
Cambia `RUIDO_W` para ver cómo se comporta con ruido (la V7 lo tolera hasta ≈2 W).

In [ ]:
RUIDO_W = 0.0
metodos = {"MSAPSO completo": make_msapso_early(Rule("none")),
           "V7 (δ=0.02)":    make_msapso_early(Rule("disp", delta=0.02, t_min=3))}
print(f"{'caso':6s} {'método':16s} {'mediciones':>10s} {'exactitud final %':>18s}")
for i, c in enumerate(scenarios()[0]):
    for nom, m in metodos.items():
        fin, ene, nm = eval_curves(m, [c], noise_w=RUIDO_W, reps=20)
        print(f"{i+1:<6d} {nom:16s} {nm[0]:10.0f} {fin[0]:18.2f}")

## 4. Más patrones: 100 sombras sintéticas
Mismo método, patrones de sombra aleatorios (misma familia que en el reporte).

In [ ]:
sint = scenarios()[1]
for nom, m in metodos.items():
    fin, ene, nm = eval_curves(m, sint, noise_w=RUIDO_W)
    print(f"{nom:16s} mediciones promedio = {nm.mean():.0f} | exactitud final promedio = {fin.mean():.2f} %")

## 5. Qué sigue
- El reporte completo (`Reporte_Corrida_V7.pdf`) tiene los resultados del simulador dinámico y la campaña de 2400 escenarios.
- `demo_interactiva_V7.html` permite ver el algoritmo paso a paso y cambiar la sombra en vivo.
- El algoritmo está en `msapso_early_V7.py` (75 líneas).